##Cleaning data

In [106]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from shapely import make_valid

In [107]:
raw_file_path = Path(
    "..",
    "Datasets",
    "ResidentPopulationbyPlanningAreaSubzoneofResidenceandTypeofDwellingCensusofPopulation2020.csv"
)



processed_folder_path = Path(
    "..",
    "Cleaned Datasets"
)
processed_file_path = processed_folder_path / "ResidentPopulationbyPlanningAreaSubzoneofResidenceandTypeofDwellingCensusofPopulation2020.csv"

print("Raw file:")
print(raw_file_path)

print("\nProcessed file:")
print(processed_file_path)

Raw file:
..\Datasets\ResidentPopulationbyPlanningAreaSubzoneofResidenceandTypeofDwellingCensusofPopulation2020.csv

Processed file:
..\Cleaned Datasets\ResidentPopulationbyPlanningAreaSubzoneofResidenceandTypeofDwellingCensusofPopulation2020.csv


In [108]:
if raw_file_path.exists():
    print("Resident Population dataset found.")
else:
    print("Resident Population dataset not found.")

Resident Population dataset found.


In [109]:
# Debug Cell (Eurison)
import os

print("Current folder:", os.getcwd())
print("Files in current folder:", os.listdir())

Current folder: c:\Users\euris\Documents\NYP\DPV Group Project\Code
Files in current folder: ['Data Cleaning Rental.ipynb', 'DataCleaningPostalDistrictGeoJSONVerified.ipynb', 'Data_Cleaning_NEA.ipynb', 'Resident_Population_cleaned.ipynb']


In [110]:
# Changed file pathing code (Eurison)
#upload the csv file and later on the geojson file to run the code
resident_population_raw = pd.read_csv(raw_file_path)

resident_population_raw.head()


,Number,Total,HDBDwellings_Total,HDBDwellings_1_and2_RoomFlats1,HDBDwellings_3_RoomFlats,HDBDwellings_4_RoomFlats,HDBDwellings_5_RoomandExecutiveFlats,CondominiumsandOtherApartments,LandedProperties,Others
0,Total,4044210,3152410,177360,575200,1338370,1061480,607940,248860,35000
1,Ang Mo Kio - Total,162280,131020,10720,55910,43030,21370,13980,15880,1390
2,Ang Mo Kio Town Centre,4810,2830,-,400,630,1800,1950,-,30
3,Cheng San,28070,27970,970,13290,8810,4900,-,-,100
4,Chong Boon,26500,26000,1470,12180,8560,3790,-,-,500


In [111]:
print("Number of rows and columns:")
print(resident_population_raw.shape)

print("\nColumn names:")
print(resident_population_raw.columns.tolist())


Number of rows and columns:
(388, 10)

Column names:
['Number', 'Total', 'HDBDwellings_Total', 'HDBDwellings_1_and2_RoomFlats1', 'HDBDwellings_3_RoomFlats', 'HDBDwellings_4_RoomFlats', 'HDBDwellings_5_RoomandExecutiveFlats', 'CondominiumsandOtherApartments', 'LandedProperties', 'Others']


As seen from the code, there are 10 columns of data in this dataset, however we are only looking for 2 colums, the region of singapore(which is in the column titled 'Number' for some reason) and the total population in that region(which is the column titled 'Total')

In [112]:
#Therefore I removed unnecessary columns first before cleaning data
resident_population_cleaned = resident_population_raw.drop([
    'HDBDwellings_Total',
    "HDBDwellings_1_and2_RoomFlats1",
    "HDBDwellings_3_RoomFlats",
    "HDBDwellings_4_RoomFlats",
    'HDBDwellings_5_RoomandExecutiveFlats',
    'CondominiumsandOtherApartments',
    'LandedProperties',
    'Others'
], axis=1)

resident_population_cleaned.head()

,Number,Total
0,Total,4044210
1,Ang Mo Kio - Total,162280
2,Ang Mo Kio Town Centre,4810
3,Cheng San,28070
4,Chong Boon,26500


In [113]:
#Renaming the columns so it makes sense
resident_population_cleaned = resident_population_cleaned.rename(columns={
    "Number": "Region",
    "Total": "Population",
})

print(resident_population_cleaned.head())

                   Region Population
0                   Total    4044210
1      Ang Mo Kio - Total     162280
2  Ang Mo Kio Town Centre       4810
3               Cheng San      28070
4              Chong Boon      26500


In [114]:
#Check what data to clean
print("Missing values:")
print(resident_population_cleaned.isnull().sum())

print("\nDuplicated region names:")
print(
    resident_population_cleaned["Region"]
    .duplicated()
    .sum()
)

Missing values:
Region        0
Population    0
dtype: int64

Duplicated region names:
0


##Matching data to the postal district map

In [115]:
# pathing cell (Eurison)
geojson_file_path = Path(
    "..",
    "Cleaned Datasets",
    "Postal_Districts_Cleaned.geojson"
)

districts = gpd.read_file(geojson_file_path)

districts[
    ["postal_district", "postal_district_name", "planning_areas"]
].head()

,postal_district,postal_district_name,planning_areas
0,D01,"Raffles Place, Cecil, Marina, People's Park",DOWNTOWN CORE | MARINA EAST | MARINA SOUTH | S...
1,D02,"Anson, Tanjong Pagar",OUTRAM
2,D03,"Queenstown, Tiong Bahru",QUEENSTOWN
3,D04,"Telok Blangah, Harbourfront",BUKIT MERAH | SOUTHERN ISLANDS
4,D05,"Pasir Panjang, Hong Leong Garden, Clementi New...",CLEMENTI


In [116]:
# Debug check (Eurison)
print(geojson_file_path.exists())

True


In [117]:
# Same as above changed pathing to raw_file_path variable earlier (Eurison)
#build a simple lookup of postal district geojson file that tells us which postal district each planning area belongs to

districts = gpd.read_file(geojson_file_path)
districts[["postal_district", "postal_district_name", "planning_areas"]].head()

,postal_district,postal_district_name,planning_areas
0,D01,"Raffles Place, Cecil, Marina, People's Park",DOWNTOWN CORE | MARINA EAST | MARINA SOUTH | S...
1,D02,"Anson, Tanjong Pagar",OUTRAM
2,D03,"Queenstown, Tiong Bahru",QUEENSTOWN
3,D04,"Telok Blangah, Harbourfront",BUKIT MERAH | SOUTHERN ISLANDS
4,D05,"Pasir Panjang, Hong Leong Garden, Clementi New...",CLEMENTI


In [118]:
area_to_district = {}
area_to_name = {}

for i in range(len(districts)):
    code = districts.loc[i, "postal_district"]
    name = districts.loc[i, "postal_district_name"]
    areas = districts.loc[i, "planning_areas"].split("|")
    for area in areas:
        area = area.strip().upper()
        area_to_district[area] = code
        area_to_name[area] = name

print("Planning areas mapped:", len(area_to_district))

Planning areas mapped: 55


In [119]:
#format column names to have same names as the postal district map
resident_population_cleaned["Population"] = resident_population_cleaned["Population"].astype(str).str.strip()
resident_population_cleaned["Population"] = resident_population_cleaned["Population"].replace("-", "0")
resident_population_cleaned["Population"] = pd.to_numeric(resident_population_cleaned["Population"])

resident_population_cleaned.head()

,Region,Population
0,Total,4044210
1,Ang Mo Kio - Total,162280
2,Ang Mo Kio Town Centre,4810
3,Cheng San,28070
4,Chong Boon,26500


In [120]:
def clean_name(text):
    text = str(text).strip()
    if text.endswith("Total"):
        text = text[:-5]
    text = text.strip()
    if text.endswith("-"):
        text = text[:-1]
    text = text.strip()
    return text
#example
print(clean_name("Ang Mo Kio - Total"))
print(clean_name("Changi- Total"))

Ang Mo Kio
Changi


In [121]:
#Tag every location/region with its postal district

planning_area_list = []
district_list = []
district_name_list = []
level_list = []

current_area = None
for region in resident_population_cleaned["Region"]:
    text = str(region).strip()
    if text == "Total":
        planning_area_list.append("ALL")
        district_list.append("ALL")
        district_name_list.append("All Singapore")
        level_list.append("country")
    elif text.endswith("Total"):
        current_area = clean_name(text)
        key = current_area.upper()
        planning_area_list.append(current_area)
        district_list.append(area_to_district[key])
        district_name_list.append(area_to_name[key])
        level_list.append("planning_area")
    else:
        key = current_area.upper()
        planning_area_list.append(current_area)
        district_list.append(area_to_district[key])
        district_name_list.append(area_to_name[key])
        level_list.append("subzone")

resident_population_cleaned["Planning_Area"] = planning_area_list
resident_population_cleaned["Postal_District"] = district_list
resident_population_cleaned["Postal_District_Name"] = district_name_list
resident_population_cleaned["Level"] = level_list

resident_population_cleaned.head()

,Region,Population,Planning_Area,Postal_District,Postal_District_Name,Level
0,Total,4044210,ALL,ALL,All Singapore,country
1,Ang Mo Kio - Total,162280,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",planning_area
2,Ang Mo Kio Town Centre,4810,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone
3,Cheng San,28070,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone
4,Chong Boon,26500,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone


In [122]:
#check for missing postal district etc
missing = resident_population_cleaned[resident_population_cleaned["Postal_District"] == ""]
print("Locations with no postal district:", len(missing))
print("Different postal districts found:", resident_population_cleaned["Postal_District"].nunique())

Locations with no postal district: 0
Different postal districts found: 29


## Save file 1: every location with its district
Keep all planning areas and subzones (drop only the whole-Singapore total row) and saves them with the new district columns.

In [123]:
population_by_location = resident_population_cleaned[resident_population_cleaned["Level"] != "country"]
population_by_location.to_csv(
    processed_folder_path / "Additional Verification Data" / "Population_by_location.csv",
    index=False
)

print("Saved Population_by_location.csv with", len(population_by_location), "rows")
population_by_location.head()

Saved Population_by_location.csv with 387 rows


,Region,Population,Planning_Area,Postal_District,Postal_District_Name,Level
1,Ang Mo Kio - Total,162280,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",planning_area
2,Ang Mo Kio Town Centre,4810,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone
3,Cheng San,28070,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone
4,Chong Boon,26500,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone
5,Kebun Bahru,22620,Ang Mo Kio,D20,"Bishan, Ang Mo Kio",subzone


## Save file 2: total population per district
This file adds up the population for each postal district (using only the planning area totals so nothing is counted twice), saves the 28 districts, then checks if the grand total still matches Singapore's total.


In [124]:
only_totals = resident_population_cleaned[resident_population_cleaned["Level"] == "planning_area"]
population_by_district = only_totals.groupby(["Postal_District", "Postal_District_Name"], as_index=False)["Population"].sum()
population_by_district = population_by_district.sort_values("Postal_District")
population_by_district.to_csv(
    processed_folder_path / "Population_By_Region_Cleaned.csv",
    index=False
)

population_by_district

,Postal_District,Postal_District_Name,Population
0,D01,"Raffles Place, Cecil, Marina, People's Park",6450
1,D02,"Anson, Tanjong Pagar",18340
2,D03,"Queenstown, Tiong Bahru",95930
3,D04,"Telok Blangah, Harbourfront",153190
4,D05,"Pasir Panjang, Hong Leong Garden, Clementi New...",91990
5,D06,"High Street, Beach Road (part)",510
6,D07,"Middle Road, Golden Mile",13120
7,D08,Little India,101290
8,D09,"Orchard, Cairnhill, River Valley",10990
9,D10,"Ardmore, Bukit Timah, Holland Road, Tanglin",21810
